In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Layer, Dense

# --------------------------------------------------
# Custom layer needed to load the saved feature extractor
# --------------------------------------------------
class SEBlock(Layer):
    def __init__(self, ratio=4, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, shape):
        ch = shape[-1]
        self.fc1 = Dense(max(ch // self.ratio, 1), activation='relu')
        self.fc2 = Dense(ch, activation='sigmoid')

    def call(self, x):
        squeeze = tf.reduce_mean(x, axis=1)
        excite = self.fc2(self.fc1(squeeze))
        return x * tf.expand_dims(excite, 1)

    def get_config(self):
        config = super().get_config()
        config.update({"ratio": self.ratio})
        return config

# --------------------------------------------------
# Paths
# --------------------------------------------------
MODEL_DIR = "/content/drive/MyDrive/models_grad_project"

FEATURE_PATH = os.path.join(MODEL_DIR, "dann_feature_extractor_all_new_29.keras")
GESTURE_PATH = os.path.join(MODEL_DIR, "dann_gesture_classifier_all_new_29.keras")

COMBINED_KERAS_PATH = os.path.join(MODEL_DIR, "dann_inference_all_new_29.keras")
TFLITE_PATH = os.path.join(MODEL_DIR, "dann_inference_all_new_29.tflite")

# --------------------------------------------------
# Load saved Keras models
# --------------------------------------------------
F = keras.models.load_model(
    FEATURE_PATH,
    custom_objects={"SEBlock": SEBlock}
)
G = keras.models.load_model(GESTURE_PATH)

print("✅ Loaded models")
print("F input shape:", F.input_shape)
print("F output shape:", F.output_shape)
print("G input shape:", G.input_shape)
print("G output shape:", G.output_shape)

# --------------------------------------------------
# Build one end-to-end inference model: input -> F -> G
# --------------------------------------------------
inp = keras.Input(shape=F.input_shape[1:], name="emg_window")
x = F(inp, training=False)
out = G(x, training=False)

inference_model = keras.Model(inp, out, name="dann_inference_model")

# Optional: save the combined Keras inference model too
inference_model.save(COMBINED_KERAS_PATH)
print(f"✅ Saved combined Keras model to: {COMBINED_KERAS_PATH}")

# --------------------------------------------------
# Convert to TFLite
# Keep float32 first for safest accuracy matching
# --------------------------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print(f"✅ Saved TFLite model to: {TFLITE_PATH}")
print(f"Model size: {len(tflite_model) / 1024:.2f} KB")

Mounted at /content/drive
✅ Loaded models
F input shape: (None, 50, 10)
F output shape: (None, 256)
G input shape: (None, 256)
G output shape: (None, 29)
✅ Saved combined Keras model to: /content/drive/MyDrive/models_grad_project/dann_inference_all_new_29.keras
Saved artifact at '/tmp/tmpw0yttywi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 10), dtype=tf.float32, name='emg_window')
Output Type:
  TensorSpec(shape=(None, 29), dtype=tf.float32, name=None)
Captures:
  138568057313552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568057313168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568057314128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568057313360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568057311440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568057312016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138568057316816: TensorSpec(shape=(), 

In [2]:
import time
import numpy as np
import tensorflow as tf

# --------------------------------------------------
# Benchmark the saved TFLite model
# --------------------------------------------------
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("✅ TFLite model loaded for benchmarking")
print("Input details:", input_details)
print("Output details:", output_details)

# Create a dummy input with the correct shape
input_shape = input_details[0]["shape"]
input_dtype = input_details[0]["dtype"]

# Example expected shape: (1, 50, 10)
dummy_input = np.random.rand(*input_shape).astype(input_dtype)

# --------------------------------------------------
# Warmup runs
# --------------------------------------------------
warmup_runs = 20
for _ in range(warmup_runs):
    interpreter.set_tensor(input_details[0]["index"], dummy_input)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]["index"])

print(f"✅ Warmup finished ({warmup_runs} runs)")

# --------------------------------------------------
# Timed runs
# --------------------------------------------------
num_runs = 200
times_ms = []

for _ in range(num_runs):
    start = time.perf_counter()

    interpreter.set_tensor(input_details[0]["index"], dummy_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])

    end = time.perf_counter()
    times_ms.append((end - start) * 1000.0)

times_ms = np.array(times_ms)

print("\n" + "=" * 60)
print("TFLite Inference Benchmark")
print("=" * 60)
print(f"Runs:               {num_runs}")
print(f"Average time:       {times_ms.mean():.3f} ms")
print(f"Median time:        {np.median(times_ms):.3f} ms")
print(f"Minimum time:       {times_ms.min():.3f} ms")
print(f"Maximum time:       {times_ms.max():.3f} ms")
print(f"Std deviation:      {times_ms.std():.3f} ms")
print(f"Inferences / second:{1000.0 / times_ms.mean():.2f}")
print("=" * 60)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


✅ TFLite model loaded for benchmarking
Input details: [{'name': 'serving_default_emg_window:0', 'index': 0, 'shape': array([ 1, 50, 10], dtype=int32), 'shape_signature': array([-1, 50, 10], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall_1:0', 'index': 89, 'shape': array([ 1, 29], dtype=int32), 'shape_signature': array([-1, 29], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
✅ Warmup finished (20 runs)

TFLite Inference Benchmark
Runs:               200
Average time:       1.499 ms
Median time:        1.444 ms
Minimum time:       0.965 ms
Maximum time:       4.100 ms
Std de